# ML Zoomcamp 2026 — Homework 5: Deploying Machine Learning Models

공식 본문: `materials/homework.md` (수정하지 않음). Official text stays in `materials/homework.md`.
각 문항 셀을 위에서부터 실행하면 제출 숫자가 출력된다. 모델은 다시 학습하지 않고 pipeline.bin 을 읽는다.
Run the question cells from the top. Each cell prints the value to submit. The model is not retrained.

- Python pin: 3.11.15 (`.python-version`)
- Frozen artifact: `pipeline.bin`
- Expected sha256: `1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1`
- Data named by the homework: `data/course_lead_scoring_2026.csv`
- Write-up: `ML_05_HW.md`
- Kernel: this folder `.venv`. Do not use `!pip` / `!wget`.
- nbclassic port **8893** (leave the server down)
- Submit: https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw05
- `homework.yaml` `due_at`: `2026-10-19T23:00:00Z`. Form already opened on 2026-10-26.
- Image tag for this assignment: `zoomcamp-model:2026-hw5`. `README.md` uses `zoomcamp-model:2026-hw10` because the same artifact is reused later.

Setup commands from the homework (notes only — do not run `smoke_test.py`, `uvicorn`, or `docker` here):

```bash
uv sync --locked
sha256sum pipeline.bin
```

Model line recorded by the homework: `DictVectorizer` then `LogisticRegression(solver="liblinear", C=1.0)`. Graded inference loads `pipeline.bin`.


In [1]:
# import os: 이 프로세스의 환경 변수를 다룬다.
import os

# os.environ.setdefault: 값이 없을 때만 넣는다. 이미 있으면 덮어쓰지 않는다.
# MPLCONFIGDIR: Matplotlib 설정 폴더. C: 사용자 폴더 대신 E: 캐시를 쓴다.
os.environ.setdefault("MPLCONFIGDIR", r"E:\IT_SPACES\AI\.cache\matplotlib")
# XDG_CACHE_HOME: 공용 캐시 루트.
os.environ.setdefault("XDG_CACHE_HOME", r"E:\IT_SPACES\AI\.cache")
# TEMP, TMP: 임시 파일. 둘 다 같은 E: tmp.
os.environ.setdefault("TEMP", r"E:\IT_SPACES\AI\.cache\tmp")
os.environ.setdefault("TMP", r"E:\IT_SPACES\AI\.cache\tmp")
# OMP/MKL/OPENBLAS/NUMEXPR: numpy·sklearn 이 코어를 전부 잡지 않게 스레드를 1로 둔다.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

# 여기까지: 라이브러리를 import 하기 전에 경로만 고정한다. pandas·sklearn 은 이 셀에 두지 않는다.
print("cache paths set")


cache paths set


## Q1 — environment check

`uv --version` 을 메모한다. 채점 항목이 아니다. 로컬 uv 버전은 모델의 속성이 아니다.
Run `uv --version` and record it in notes. This is a setup check, not a graded answer.

```bash
uv --version
```

**Answer:** 채점 안 함. 이 PC에서 `uv --version` 은 8초 안에 끝나지 않아 버전 숫자는 비움.


In [2]:
# import subprocess: 외부 명령을 실행하고 출력을 받는다.
import subprocess

# subprocess.run: 과제 1번은 uv --version 이다. 채점 대상은 아니다.
# timeout=8: 이 PC에서 uv 가 멈추면 커널이 같이 멈추므로 8초에서 끊는다.
# capture_output=True, text=True: 버전 문장을 문자열로 받는다.
try:
    completed = subprocess.run(
        ["uv", "--version"],
        capture_output=True,
        text=True,
        timeout=8,
    )
    # stdout 이 비면 stderr 를 보여 준다.
    print((completed.stdout or completed.stderr).strip())
except subprocess.TimeoutExpired:
    # 여기까지: 버전 숫자를 못 받아도 다음 문항으로 넘어간다.
    print("uv --version timed out")


uv 0.13.5 (46776a95d 2026-10-21 x86_64-pc-windows-msvc)


## Q2 — locked dependency

`pyproject.toml` 과 `uv.lock` 에 들어 있는 2026 reference Scikit-Learn 버전. 락 파일을 다시 만들지 말고 `uv sync --locked` 를 쓴다.
Which Scikit-Learn version is part of the 2026 reference environment? Do not regenerate the lockfile.

- `1.6.1`
- `1.7.2`
- `1.8.0`
- `2.0.0`

```bash
uv sync --locked
```

**Answer:** 1.7.2


In [3]:
# import re: 파일 안에서 버전 문자열을 찾는다.
import re
# Path: pyproject.toml 과 uv.lock 의 경로.
from pathlib import Path

# cwd: 노트북을 HW_05 폴더에서 열었을 때의 작업 폴더.
root = Path.cwd()
# read_text: 락을 다시 만들지 않고, 이미 있는 선언만 읽는다.
project_text = (root / "pyproject.toml").read_text(encoding="utf-8")
lock_text = (root / "uv.lock").read_text(encoding="utf-8")

# pyproject.toml 의 의존성 줄은 scikit-learn==1.7.2 형태다.
project_match = re.search(r"scikit-learn==(\d+\.\d+\.\d+)", project_text)
# uv.lock 의 [[package]] 블록은 name 다음 줄에 version 이 있다.
lock_match = re.search(
    r'name = "scikit-learn"\s+version = "(\d+\.\d+\.\d+)"',
    lock_text,
)
options = ["1.6.1", "1.7.2", "1.8.0", "2.0.0"]
pinned = project_match.group(1) if project_match else None
lock_version = lock_match.group(1) if lock_match else None

# print: 두 파일이 같은 버전을 가리키는지, 그리고 보기 중 무엇인지.
print("pyproject", pinned)
print("uv.lock", lock_version)
print("choice", pinned if pinned in options and pinned == lock_version else None)


pyproject 1.7.2
uv.lock 1.7.2
choice 1.7.2


## Q3 — load the model

`pipeline.bin` 을 `pickle` 로 읽고, 아래 paid_ads 리드에 `predict_proba` 를 호출한다. 전환 확률은 소수 셋째 자리. 허용 오차 `0.005`. 보기에서 고르지 않는다. 셀이 소수 셋째 자리를 출력한다.
Load `pipeline.bin` with `pickle` and call `predict_proba` for this paid-ads lead. Report the conversion probability to three decimal places. Tolerance `0.005`. Do not pick from a list.

```json
{
  "lead_source": "paid_ads",
  "industry": "technology",
  "employment_status": "employed",
  "location": "north_america",
  "number_of_courses_viewed": 2,
  "annual_income": 79276.0,
  "interaction_count": 4,
  "lead_score": 0.41
}
```

**Answer:** 0.533


In [4]:
# import pickle: pipeline.bin 을 파이썬 객체로 되돌린다.
import pickle
# Path: 모델 파일 위치.
from pathlib import Path

# 여기까지: 이 import 가 커널에서 처음이면 numpy·sklearn DLL 을 읽느라 길다. 같은 커널에서 다시 실행하면 짧다.
# sklearn 은 공유 venv 정션(ML\.venv)에 있는 버전을 쓴다. 피클은 1.7.2 로 저장돼 있어 경고가 날 수 있다.
model_path = Path("pipeline.bin")
# open rb: 이진 파일을 읽기 전용으로 연다.
with model_path.open("rb") as model_file:
    pipeline = pickle.load(model_file)

# 과제 3번 리드. 키 이름은 학습 때 쓴 특성 이름과 같다.
lead_q3 = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}
# predict_proba: 각 클래스 확률. [0, 1] 은 전환(클래스 1) 확률이다.
probability_q3 = float(pipeline.predict_proba([lead_q3])[0, 1])
# round 3: 제출은 소수 셋째 자리. 허용 오차는 0.005.
print(round(probability_q3, 3))
print("raw", probability_q3)


E:\IT_SPACES\AI\ZoomCamp\ML\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


0.533
raw 0.5329944186432507


E:\IT_SPACES\AI\ZoomCamp\ML\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
E:\IT_SPACES\AI\ZoomCamp\ML\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## Q4 — serve the model

제공된 API 를 띄운 뒤 organic_search 리드를 `POST /predict` 로 보낸다. `conversion_probability` 는 소수 셋째 자리. 응답에는 `[0, 1]` 확률과 boolean `conversion` 이 있다. 이 스캐폴드에서는 서버를 실행하지 않는다.
Start the API and POST the organic-search lead. Report `conversion_probability` to three decimal places. Same `0.005` tolerance. Do not start the server in this scaffold.

```bash
uv run uvicorn predict:app --host 0.0.0.0 --port 9696
```

```bash
curl -s http://localhost:9696/predict \
  -H 'Content-Type: application/json' \
  -d '{
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74
  }'
```

**Answer:** 0.770


In [5]:
# Q3 에서 올린 pipeline 이 없으면 같은 피클을 다시 읽는다.
import pickle
from pathlib import Path

if "pipeline" not in dir():
    with Path("pipeline.bin").open("rb") as model_file:
        pipeline = pickle.load(model_file)

# 과제 4번 리드. API 의 POST /predict 가 이 딕셔너리에 predict_proba 를 호출한다.
lead_q4 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}
probability_q4 = float(pipeline.predict_proba([lead_q4])[0, 1])
# predict.py 는 확률을 소수 여섯 자리로 넣고, conversion 은 0.5 이상인지다.
# 제출 칸은 소수 셋째 자리다.
print(round(probability_q4, 3))
print("raw", probability_q4)
print("conversion", probability_q4 >= 0.5)


0.77
raw 0.7697992923369713
conversion True


## Q5 — container base image

Dockerfile 에 선언된 Python base-image 태그. 로컬 이미지 크기는 채점하지 않는다. 이 과제의 빌드 태그는 `zoomcamp-model:2026-hw5` 다. README 의 `zoomcamp-model:2026-hw10` 은 같은 아티팩트를 나중에 다시 쓰기 때문이다.
Which Python base-image tag is declared in the Dockerfile? Local image size is not graded. Assignment tag is `zoomcamp-model:2026-hw5`.

- `python:3.9-slim-bullseye`
- `python:3.11.15-slim-bookworm`
- `python:3.12-slim-bookworm`
- `python:3.13.10-slim-bookworm`

```bash
docker build -t zoomcamp-model:2026-hw5 .
docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5
```

**Answer:** python:3.11.15-slim-bookworm


In [6]:
# Path: Dockerfile 은 이 폴더에 있다. 이미지를 빌드하지 않고 선언만 읽는다.
from pathlib import Path

# read_text: 첫 줄이 FROM python:...@sha256:... 이다.
dockerfile = (Path("Dockerfile")).read_text(encoding="utf-8")
from_line = dockerfile.splitlines()[0]
# split: 태그에서 다이제스트(@sha256) 앞부분만 남긴다.
image = from_line.split()[1].split("@")[0]
options = [
    "python:3.9-slim-bullseye",
    "python:3.11.15-slim-bookworm",
    "python:3.12-slim-bookworm",
    "python:3.13.10-slim-bookworm",
]
# print: 보기와 일치하는 태그가 답이다. 로컬 이미지 크기는 채점하지 않는다.
print(image)
print("choice", image if image in options else None)


python:3.11.15-slim-bookworm
choice python:3.11.15-slim-bookworm


## Q6 — run the container

Q4 와 같은 요청을 컨테이너에 보낸다. 제공 클라이언트는 `q6_test.py`. `conversion_probability` 는 소수 셋째 자리이고 Q4 와 같은 기준 추론이다. `/health` 는 `{"status":"ok", ...}` 와 `model_metadata.json` 의 모델 체크섬을 돌려준다. 이 스캐폴드에서는 `q6_test.py` 와 Docker 를 실행하지 않는다.
Send the Question 4 request to the container. Report `conversion_probability` to three decimal places. Do not run `q6_test.py` or Docker in this scaffold.

```bash
python q6_test.py
```

```bash
curl -s http://localhost:9696/health
```

**Answer:** 0.770


In [7]:
# Q6 는 컨테이너에 Q4 와 같은 요청을 보낸다. 과제 본문은 그 확률이 Q4 와 같다고 한다.
# Docker 를 띄우지 않고, Q4 와 같은 predict_proba 를 다시 계산한다.
import json
import pickle
from pathlib import Path

if "pipeline" not in dir():
    with Path("pipeline.bin").open("rb") as model_file:
        pipeline = pickle.load(model_file)

lead_q4 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}
probability_q6 = float(pipeline.predict_proba([lead_q4])[0, 1])
# q6_test.py 가 POST 로 받는 필드와 같다.
print(round(probability_q6, 3))

# model_metadata.json: /health 가 돌려주는 model_sha256 의 기준 값.
meta = json.loads(Path("model_metadata.json").read_text(encoding="utf-8"))
print("health model_sha256", meta["model_sha256"])


0.77
health model_sha256 1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1
